In [186]:
from IPython.display import HTML, Javascript, display

display(HTML("""
<style>
a[href^="http://"],
a[href^="https://"] {
    color: blue !important;
}

/* stil za sve tabele */
table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important;
    max-width: none !important;
    table-layout: auto !important;
    border-collapse: collapse !important;
    font-family: "Segoe UI", "Calibri", sans-serif !important;
    font-size: 12px !important;
}

th, td {
    padding: 8px 20px !important;
}

/* markdown tabela – zadrži staro ponašanje */
table:not(.dataframe) tr:first-child th {
    border-bottom: 2px solid #222 !important;
}

/* dataframe – samo pravo zaglavlje */
table.dataframe thead tr:first-child th {
    border-bottom: 2px solid #222 !important;
}

/* index kolona dataframe da ne bude bold */
table.dataframe tbody th {
    font-weight: normal !important;
}

/* linija ispod poslednjeg reda */
tbody tr:last-child td,
tbody tr:last-child th {
    border-bottom: 1px solid #aaa !important;
}

/* caption */
caption,
.table-caption,
p.caption,
div.caption,
figcaption {
    text-align: center !important;
    font-weight: bold !important;
}
</style>
"""))

display(Javascript("""
function promeniTableUTabela() {
    const selectors = [
        "caption",
        ".table-caption",
        "p.caption",
        "div.caption",
        "figcaption",
        ".caption-number"
    ];

    document.querySelectorAll(selectors.join(",")).forEach(function(el) {
        if (el.innerHTML.includes("Table")) {
            el.innerHTML = el.innerHTML.replace(/\\bTable\\b/g, "Tabela");
        }
    });
}

promeniTableUTabela();

/* ako se notebook kasnije rerenderuje */
const observer = new MutationObserver(function() {
    promeniTableUTabela();
});

observer.observe(document.body, {
    childList: true,
    subtree: true
});
"""))

<IPython.core.display.Javascript object>

# Regularni izrazi 

Regularni izrazi (regular expressions, regex) su alat za rad sa tekstom koji omogućava pretragu, prepoznavanje i obradu obrazaca u stringovima. Predstavljaju „jezik za opisivanje teksta“ — umesto da ručno prolazimo kroz karaktere, definišemo šablon koji automatski pronalazi ono što tražimo. Ako standardne string operacije rade nad konkretnim tekstom, regularni izrazi omogućavaju rad nad celim klasama obrazaca (npr. svi brojevi, email adrese ili datumi). U Pajtonu se implementiraju pomoću biblioteke `re`.

**Ishodi poglavlja**: Nakon proučavanja ovog poglavlja student će biti u stanju da:
- definiše osnovne regex obrasce za pretragu teksta,
- koristi funkcije biblioteke re za pronalaženje, izdvajanje i zamenu delova stringa,
- konstruiše složenije obrasce za validaciju i obradu realnih tekstualnih podataka.

### Uvod 

Verovatno ste već koristili pretragu teksta pomoću kombinacije tastera *CTRL-F*, gde unosite tačnu reč koju želite da pronađete. Regularni izrazi idu korak dalje — omogućavaju da ne tražimo samo konkretnu reč, već čitav obrazac teksta. Na primer, možda ne znate tačan broj telefona neke firme, ali znate njegovu strukturu: tri cifre, zatim crtica, pa još nekoliko cifara. Kao ljudi, lako prepoznajemo da je *064-555-1234* validan broj, dok *4.155.551.234* to nije.

Slično tome, svakodnevno prepoznajemo i druge obrasce: email adrese sadrže znak @, URL adrese imaju tačke i kose crte, hashtagovi počinju znakom # i ne sadrže razmake, dok različiti identifikacioni brojevi imaju tačno definisanu strukturu. Regularni izrazi omogućavaju da takva „intuicija o obrascima“ bude formalizovana i primenjena u programiranju.

Iako su izuzetno korisni, regularni izrazi su relativno slabo poznati van programerskog sveta, uprkos tome što ih mnogi alati — poput Microsoft Word ili OpenOffice — već podržavaju u funkcijama pretrage i zamene. Njihova snaga ogleda se u tome što omogućavaju da se složeni problemi rešavaju brzo i elegantno, često uz minimalan broj koraka.

U ovom poglavlju najpre ćemo videti kako se obrasci u tekstu mogu pronalaziti bez regularnih izraza, a zatim kako se isti problemi rešavaju pomoću njih — znatno kraće i efikasnije. Upoznaćemo osnovne mehanizme pretrage, kao i naprednije tehnike poput zamene teksta i definisanja sopstvenih klasa karaktera. Na kraju, razvićemo program koji automatski prepoznaje i izdvaja podatke poput brojeva telefona i email adresa iz teksta.

### Pronalaženje obrazaca teksta bez regularnih izraza

Recimo da želite da pronađete telefonski broj u stringu. Znate obrazac ako ste iz Srbije: tri broja, crtica, tri broja, crtica i četiri broja. 
Evo primera: 064-234-4242. Koristimo funkciju nazvanu `tel_broj()` da proverimo da li se string podudara sa ovim šablonom, vraćajući `True` ili `False`.

In [5]:
def tel_broj(text):
    if len(text) != 12:
        return False
    for i in range(0, 3):
        if not text[i].isdecimal():
            return False
    if text[3] != '-':
        return False
    for i in range(4, 7):
        if not text[i].isdecimal():
            return False
    if text[7] != '-':
        return False
    for i in range(8, 12):
         if not text[i].isdecimal():
            return False
    return True

In [11]:
print('Da li je 064-234-4142 telefonski broj?')
print(tel_broj('064-234-4142'))

Da li je 064-234-4142 telefonski broj?
True


In [7]:
print('Da li je Petar Petrović telefonski broj?')
print(tel_broj('Petar Petrović'))

Da li je Petar Petrović telefonski broj?
False


Ako želite da pronađete telefonski broj u većem stringu, moraćete da dodate još više koda da biste pronašli obrazac telefonskog broja:

In [10]:
poruka = 'Pozovi me sutra na broj telefona 063-555-1011. Moj broj telefona u kancelariji je 011-555-9999.'
for i in range(len(poruka)):
    deo = poruka[i:i+12]
    if tel_broj(deo):
        print('Pronađen broj telefona: ' + deo)
print('Gotovo')

Pronađen broj telefona: 063-555-1011
Pronađen broj telefona: 011-555-9999
Gotovo


Iako je string u poruci u ovom primeru kratak, on bi mogao biti dugačak milion znakova, a program bi i dalje radio za manje od sekunde. Sličan program koji 
pronalazi brojeve telefona pomoću regularnih izraza takođe bi se pokrenuo za manje od sekunde, ali regularni izrazi ubrzavaju pisanje ovih programa.

#### Pronalaženje obrazaca teksta sa regularnim izrazima

Prethodni program za pronalaženje telefonskog broja funkcioniše, ali koristi mnogo koda da bi uradio nešto ograničeno: funkcija `tel_broj()` ima 17 redova, ali može pronaći samo jedan obrazac brojeva telefona. Šta je sa brojem telefona formatiranim poput 064.345.1212 ili (64)234233? Šta ako telefonski broj ima dodatak, na primer +381-64-4242-234? Funkcija `tel_broj()` neće uspeti da ih potvrdi. Možete dodati još koda za ove dodatne obrasce, ali postoji lakši način.

Regularni izrazi, skraćeno nazvani *regexes*, opisi su obrasca teksta. Na primer, a `\d` u regularnom izrazu označava cifreni znak - to jest bilo koji pojedinačni broj od 0 do 9. Redovni izraz `\d\d\d-\d\d\d-\d\d\d\d` koristi Pithon za podudaranje sa istim tekstualnim obrascem kao i prethodna funkcija `is_phone_number()`: string od tri broja, crtica, još tri broja, još jedna crtica i četiri broja. Bilo koji drugi string ne bi se podudarao sa `\d\d\d-\d\d\d-\d\d\d\d` regularnim izrazom.

Ali regularni izrazi mogu biti mnogo sofisticiraniji. Na primer, dodavanje mesta 3 u zagradama ({3}) nakon uzorka je kao da kažete: „Poravnajte ovaj obrazac tri puta. Dakle, nešto kraći regularni izraz `\d{3}-\d{3}-\d{4}` takođe se podudara sa ispravnim formatom telefonskog broja.

### Kreiranje Regex objekata

Sve regex funkcije u Pajtonu se nalaze u modulu `re`.

In [13]:
import re

Ako se prosledi vrednost stringa koji predstavlja vaš regularni izraz funkciji `re.compile()` ona vraća objekat `Regex` šablona (ili jednostavno `Regex` objekat).


Da biste kreirali `Regex` objekat koji se podudara sa uzorkom (šablonom) telefonskog broja, unesite sledeće: 
(Zapamtite da `\d` znači „cifreni znak“, a `\d\d\d-\d\d\d-\d\d\d\d` je regularni izraz za obrazac telefonskog broja.)

In [14]:
tel_broj_regex = re.compile(r'\d\d\d-\d\d\d-\d\d\d\d')

Sada `tel_broj_regex` promenljiva sadrži `Regex` objekat.

#### Uparivanje Regex objekata

Metoda `search()` `Regex` objekta pretražuje string koji mu je prosleđen za bilo koje podudaranje sa regularnim izrazom. Metoda `search()` će vratiti `None` ako obrazac regularnog izraza nije pronađen u stringu. Ako je obrazac pronađen, metoda `search()` vraća objekt `Match`, koji ima metodu `group()` koja će vratiti stvarni podudarni tekst iz pretraživanog stringa. (Ubrzo ću objasniti grupe.) Na primer, unesite sledeće:

In [16]:
tel_broj_Regex = re.compile(r'\d\d\d-\d\d\d-\d\d\d\d')
tel = tel_broj_Regex.search('Moj broj telefona je 062-534-2242.')
print('Pronađen broj telefona: ' + tel.group())

Pronađen broj telefona: 062-534-2242


Promenljiva `tel` je ime za `Match` objekte. Ovaj primer možda izgleda komplikovano na početku, ali je mnogo kraći od prethodno napisanih programa.

Ovde prosleđujemo željeni obrazac `re.compile()` i rezultirajući `Regex` objekat čuvamo u `tel_broj_Regex`. Zatim pozivamo `search()` na `tel_broj_Regex` i prenosimo `search()` string za koji želimo da ispitamo podudaranje tokom pretraživanja. Rezultat pretrage čuva se u promenljivoj `tel`. U ovom primeru znamo da će se naš obrazac naći u stringu, pa znamo da će se vratiti objekt `Match`. Znajući da `tel` sadrži objekt podudaranja, a ne `null` vrednost `None`, možemo pozvati `group()` na `tel` da vratimo podudaranje. Pisanje `tel.group()` unutar našeg poziva funkcije `print()` prikazuje celo podudaranje, 062-555-4242.

#### Pregled podudaranja regularnih izraza

Iako postoji nekoliko koraka za upotrebu regularnih izraza u Pithonu, svaki korak je prilično jednostavan

1. Uvezite regex modul sa `import re`.
2. Stvorite `Regex` objekat pomoću funkcije `re.compile()`. (Ne zaboravite da koristite sirovi string.) 
3. Prenesite string koji želite da pretražite metodi `search()` `Regex` objekta. Ona vraća objekt podudaranje `Match`. 
4. Pozovite metod `group()` objekta `Match` da biste vratili string stvarnog podudarnog teksta.

### Još o uparivanju šablona sa regularnim izrazima

Sada kada znate osnovne korake za kreiranje i pronalaženje objekata regularnih izraza pomoću Pithona, spremni ste da isprobate neke od njihovih moćnijih mogućnosti 
usklađivanja uzoraka/šablona.

#### Grupisanje sa zagradama

Recimo da želite da odvojite pozivni broj od ostatka telefonskog broja. Dodavanjem zagrada stvoriće se grupe u regularnom izrazu: `(\d\d\d)-(\d\d\d-\d\d\d\d)`. 
Tada možete koristiti metodu objekta podudaranja `group()` da biste uzeli odgovarajući tekst iz samo jedne grupe.

Prvi skup zagrada u nizu regularnog izraza biće grupa 1. Drugi skup biće grupa 2. Prenošenjem celog broja 1 ili 2 na metodu objekta podudaranja `group()` možete da 
preuzmete različite delove podudarnog teksta. Prosleđivanje 0 ili ništa metodi `group()` vratiće ceo podudarni tekst. Unesimo sledeće:

In [19]:
tel_num_Regex = re.compile(r'(\d\d\d)-(\d\d\d-\d\d\d\d)')
tel = tel_num_Regex.search('Moj broj telefona je 065-523-2421.')

In [25]:
tel.group(1)

'065'

In [26]:
tel.group(2)

'523-2421'

In [22]:
tel.group(0) 

'065-523-2421'

In [23]:
tel.group() 

'065-523-2421'

Ako želite da preuzmete sve grupe odjednom, koristite metodu `groups()` - obratite pažnju na oblik množine za ime.

In [24]:
tel.groups()

('065', '523-2421')

In [27]:
provajder, glavniBroj = tel.groups()

In [29]:
print(provajder)

065


In [30]:
print(glavniBroj)

523-2421


Budući da `tel.groups()` vraća više vrednosti, trikom višestrukog dodeljivanja možete da dodelite svaku vrednost zasebnoj promenljivoj, kao u prethodnom redu.

Zagrade imaju posebno značenje u regularnim izrazima, ali šta radite ako treba da podudarite zagradu u svom tekstu? Na primer, možda telefonski brojevi kojima pokušavate da se podudaraju imaju pozivni broj u zagradama. U ovom slučaju morate da izbegnete znakove `(` i `)` pomoću kose crte. Unesite sledeće

In [32]:
tel_broj_Regex = re.compile(r'(\(\d\d\d\)) (\d\d\d-\d\d\d\d)')
tel = tel_broj_Regex.search('Moj broj telefona je (060) 555-4242.')

In [33]:
tel.group(1)

'(060)'

In [34]:
tel.group(2)

'555-4242'

Znakovi za izlazak `(` i `)` u sirovom stringu prosleđenom u `re.compile()` podudaraće se sa stvarnim znakovima u zagradama. U regularnim izrazima, sledeći znakovi 
imaju posebna značenja: ` .  ^  $  *  +  ?  {  }  [  ]  \  |  (  )`

Ako želite da otkrijete ove znakove kao deo svog tekstualnog obrasca, morate da ih ispisujete kosom crtom: </br> \\.  \\^  \\$  \\*  \\+  \\?  \\{  \\}  \\[  ... 

Obavezno dvaput proverite da li ste u regularnom izrazu pogrešno napisali zagrade `\(` i `\)` i zagrade `(` i `)`. Ako dobijete poruku o grešci o „nedostajućim“ ili „neuravnoteženim zagradama“, možda ste zaboravili da uključite zatvarajući nepregrađene zagrade za grupu, kao u ovom primeru:

In [20]:
re.compile(r'(\(Zagrade\)')

error: missing ), unterminated subpattern at position 0

Poruka o grešci govori vam da postoji početna zagrada u indeksu 0 stringa `'r(\ (zagrade \)'` kojem nedostaju odgovarajuće zagrade u zatvaranju.

#### Povezivanje više grupa sa cevovodom (pipe)

Karakter `|` naziva se cev/lula. Možete ga koristiti bilo gde gde želite da se podudara sa jednim od mnogih izraza. Na primer, regularni izraz `r'Marija|Azdejković'` podudaraće se sa `'Marija'` ili `'Azdejković'`.

Kada se i `Marija` i `Azdejković` pojave u pretraživanom stringu, prva pojava odgovarajućeg teksta vratiće se kao objekt Match. Unesimo sledeće:

In [36]:
familyRegex = re.compile(r'Marija|Aleksandar')

In [37]:
m1 = familyRegex.search('Marija i Aleksandar')

In [38]:
m1.group()

'Marija'

In [40]:
m2 = familyRegex.search('Aleksandar i Marija')

In [41]:
m2.group()

'Aleksandar'

*Primedba:* Možemo pronaći sva podudaranja metodom `findall()`, ali o tome kasnije.

Takođe možete koristiti cev da utvrdite podudaranje sa jednim od nekoliko obrazaca kao deo vašeg regularnog izraza. Na primer, recimo da želite da pronađete podudaranje sa bilo  kojim stringom „Batman“, „Batmobile“, „Batcopter“ i „Batbat“. Budući da svi ovi nizovi počinju sa Bat, bilo bi lepo kada biste taj prefiks mogli odrediti samo jednom. To se može uraditi pomoću zagrada. Unesite sledeće:

In [44]:
batRegex = re.compile(r'Bat(man|mobile|copter|bat)')

In [45]:
m1 = batRegex.search('Batmobile lost a wheel')

In [46]:
m1.group()

'Batmobile'

In [47]:
m1.group(1)

'mobile'

Poziv metode `m1.group()` vraća puni podudarni tekst `'Batmobile'`, dok `m1.group(1)` vraća samo deo podudarnog teksta unutar prve grupe zagrada, `'mobile'`. Upotrebom znaka lule i grupisanjem zagrada možete odrediti nekoliko alternativnih obrazaca za koje želite da se vaš regularni izraz podudara.

Ako tražite podudaranje sa stvarnim znakom cevi, izbegnite ga kosom crtom, poput `\|`.

#### Opciono uparivanje sa znakom pitanja

Ponekad postoji obrazac za koji želite da se podudara samo opciono. Odnosno, regularni izraz treba da pronađe podudaranje bez obzira na to da li je taj deo teksta 
tamo. Znak ? označava grupu koja mu prethodi kao opcioni deo šablona. Na primer, ako unesemo sledeće:

In [48]:
batRegex = re.compile(r'Bat(wo)?man')

In [49]:
mo1 = batRegex.search('The Adventures of Batman')

In [50]:
mo1.group()

'Batman'

In [51]:
mo2 = batRegex.search('The Adventures of Batwoman')

In [52]:
mo2.group()

'Batwoman'

Deo `(wo)?` deo regularnog izraza znači da je obrazac `wo` neobavezna grupa. Redovni izraz će odgovarati tekstu koji sadrži nula instanci ili jednu instancu `wo`. 
Zbog toga se regularni izraz podudara i sa „Batwoman“ i „Batman“.

Koristeći raniji primer broja telefona, možete da naterate regularni izraz da traži brojeve telefona koji imaju ili nemaju pozivni broj. Unesimo sledeće:

In [53]:
telRegex = re.compile(r'(\d\d\d-)?\d\d\d-\d\d\d\d')

In [54]:
mo1 = telRegex.search('Moj broj telefona je 065-342-4242')

In [55]:
mo1.group()

'065-342-4242'

In [56]:
mo2 = telRegex.search('Moj broj telefona je 513-4242')

In [57]:
mo2.group()

'513-4242'

Znak `?` možete zamisliti kao što je rečeno: „Uparite nijednu ili jednu grupu koja prethodi ovom znaku pitanja.“

Ako vam treba podudaranje stvarnog znaka pitanja, to činite se sa `\?`.

#### Nula ili više uparivanja sa zvezdicom

Znak `*` (zvezdica) znači „podudaranje nula ili više puta“ - grupa koja prethodi zvezdici može se pojaviti bilo koji broj puta u tekstu. Može biti potpuno odsutan ili ponavljati iznova i iznova. Pogledajmo ponovo primer Betmena.

In [58]:
batRegex = re.compile(r'Bat(wo)*man')

In [59]:
mo1 = batRegex.search('The Adventures of Batman')

In [60]:
mo1.group()

'Batman'

In [61]:
mo2 = batRegex.search('The Adventures of Batwoman')

In [62]:
mo2.group()

'Batwoman'

In [63]:
mo3 = batRegex.search('The Adventures of Batwowowowoman')

In [64]:
mo3.group()

'Batwowowowoman'

Za „Batman“, `(wo)*` deo regularnog izraza podudara se sa nula instanci `wo` u stringu; za „Batwoman“, `(wo)*` se podudara sa jednom instancom `wo`; a za „Batwowowowoman“, `(wo)*` odgovara četiri slučaja `wo`.

Ako trebate podudarati stvarni znak zvezde, prefiksu zvezde u regularnom izrazu dodajte kosu kosu crtu, `\*`.

#### Jedno ili više uparivanja sa plusom

Dok `*` znači „podudaranje nula ili više puta“, `+` (ili plus) znači „podudaranje jednom ili više puta“. Za razliku od zvezde, koja ne zahteva da se njena grupa pojavljuje u podudarnom stringu, grupa koja prethodi plusu mora se pojaviti najmanje jednom. Nije neobavezna. Unesite sledeće i uporedite regeksima sa zvezdom u prethodnom odeljku:

In [65]:
batRegex = re.compile(r'Bat(wo)+man')

In [66]:
mo1 = batRegex.search('The Adventures of Batwoman')

In [67]:
mo1.group()

'Batwoman'

In [68]:
mo2 = batRegex.search('The Adventures of Batwowowowoman')

In [69]:
mo2.group()

'Batwowowowoman'

In [70]:
mo1 = batRegex.search('The Adventures of Batman')

In [71]:
mo1 == None

True

Regex `Bat(wo)+` se neće podudarati sa stringom „The Adventures of Batman“, jer je za znak plus potreban bar jedno pojavljivanje `wo`.

Ako trebate podudarati sa stvarnim znakom plusa, prefiksujte znak plus sa kosom crtom: `\+`.

#### Uparivanje specifičan broj puta sa zagradama

Ako imate grupu koju želite da ponovite određeni broj puta, sledite grupu u svom regularnom izrazu sa brojem u zagradama. Na primer, regularni izraz `(Ha){3}` će se 
podudarati sa stringom „HaHaHa“, ali se neće podudarati sa „HaHa“, jer poslednji ima samo dva ponavljanja iz grupe (Ha).

Umesto jednog broja, možete odrediti opseg tako što ćete između zagrada napisati minimum, zarez i maksimum. Na primer, regularni izraz `(Ha){3,5}` će se podudarati 
sa „HaHaHa“, „HaHaHaHa“ i „HaHaHaHaHa“. 

Takođe možete izostaviti prvi ili drugi broj u zagradama da biste ostavili minimum ili maksimum bez ograničenja. Na primer, `(Ha){3,}` će se podudarati sa tri ili 
više primeraka grupe (Ha), dok će se `(Ha){,5}` podudarati sa nula do pet primeraka. Zagrade mogu pomoći da vaši regularni izrazi budu kraći. Ova dva regularna 
izraza se podudaraju sa identičnim obrascima:

`(Ha){3}`  

`(Ha)(Ha)(Ha)`

I ova dva regularna izraza se takođe podudaraju sa identičnim obrascima:

`(Ha){3,5}`

`((Ha)(Ha)(Ha))|((Ha)(Ha)(Ha)(Ha))|((Ha)(Ha)(Ha)(Ha)(Ha))`

Unesimo sledeće:

In [72]:
 haRegex = re.compile(r'(Ha){3}')

In [73]:
mo1 = haRegex.search('HaHaHa')

In [74]:
mo1.group()

'HaHaHa'

In [75]:
mo2 = haRegex.search('Ha')

In [76]:
mo2 == None

True

Ovde se `(Ha){3}` podudara sa „HaHaHa“, ali ne i sa „Ha“. Pošto se ne podudara sa „Ha“, `search()` vraća `None`.

#### Pohlepno i nepohlepno podudaranje

Budući da se `(Ha){3,5}` može podudarati sa tri, četiri ili pet primeraka `Ha` u nizu 'HaHaHaHaHa', možda se pitate zašto poziv objekta `Match` u `group()` 
u prethodnom primeru zagrade vraća 'HaHaHaHaHa' umesto kraće mogućnosti. Na kraju, „HaHaHa“ i „HaHaHaHa“ su takođe važeća podudaranja regularnog izraza `(Ha){3,5}`.

Pithonovi regularni izrazi su podrazumevano pohlepni, što znači da će se u dvosmislenim situacijama podudarati sa najdužim mogućim stringom. Nepohlepna 
(takođe nazvana lenja) zagrada, koja se podudara sa najkraćim mogućim stringom, ima završnu zagradu praćenu znakom pitanja.

Unesite sledeće i uočite razliku između pohlepnih i ne pohlepnih oblika zagrada koje pretražuju isti string:

In [77]:
greedyHaRegex = re.compile(r'(Ha){3,5}')

In [78]:
mo1 = greedyHaRegex.search('HaHaHaHaHa')

In [79]:
mo1.group()

'HaHaHaHaHa'

In [80]:
nongreedyHaRegex = re.compile(r'(Ha){3,5}?')

In [81]:
mo2 = nongreedyHaRegex.search('HaHaHaHaHa')

In [82]:
mo2.group()

'HaHaHa'

Imajte na umu da upitnik može imati dva značenja u regularnim izrazima: proglašavanje ne pohlepnog podudaranja ili označavanje opcionalne grupe. Ova značenja su 
potpuno nepovezana.

#### Metod `findall()`

Pored metode `search()`, Regex objekti imaju i metod `findall()`. Dok će `search()` vratiti objekt podudaranja prvog podudarnog teksta u pretraživanom string, 
metod `findall()` vratit će stringove svakog podudaranja u pretraživanom stringu. Da biste videli kako `search()` vraća objekt podudaranja samo na prvoj instanci 
podudarnog teksta, unesite sledeće u radnu ćeliju:

In [83]:
telRegex = re.compile(r'\d\d\d-\d\d\d-\d\d\d\d')

In [84]:
mo = telRegex.search('Mobilni: 069-521-1234 Posao: 011-512-4321')

In [85]:
mo.group()

'069-521-1234'

S druge strane, `findall()` neće vratiti objekt `Match` već listu stringova - *sve dok u regularnom izrazu nema grupa*. Svaki string na listi je deo pretraživanog 
teksta koji se podudara sa regularnim izrazom. U interaktivn ćeliju unesite sledeće:

In [86]:
telRegex = re.compile(r'\d\d\d-\d\d\d-\d\d\d\d') # nema grupa

In [87]:
telRegex.findall('Cell: 415-555-9999 Work: 212-555-0000')

['415-555-9999', '212-555-0000']

Ako u regularnom izrazu postoje grupe, tada će `findall()` vratiti listu n-torki. Svaka n-torka predstavlja pronađeno podudaranje, a njene stavke su usklađeni 
stringovi za svaku grupu u regularnom izrazu. Da biste videli funkciju `findall()`, u interaktivnu ćeliju unesite sledeće (imajte na umu da regularni izraz 
koji se kompajlira sada ima grupe u zagradama):

In [88]:
telRegex = re.compile(r'(\d\d\d)-(\d\d\d)-(\d\d\d\d)') # ima grupa

In [89]:
telRegex.findall('Cell: 415-555-9999 Work: 212-555-0000')

[('415', '555', '9999'), ('212', '555', '0000')]

Da rezimiramo šta metoda `findall()` vraća, upamtite sledeće:

- Kada se pozove regularni izraz bez grupa, kao što je `\d\d\d-\d\d\d-\d\d\d\d`, metoda `findall()` vraća listu uparenih stringova, kao što je `['011-512-7878', '021-123-4321']`. 
- Kada se pozove na regularni izraz koji ima grupe, kao što je `(\d\d\d)-(\d\d\d)-(\d\d\d\d)`, metoda `findall()` vraća listu nizova (po jedan niz za svaku grupu), kao što je `[('011', '512', '7878'), ('021', '123', '4321')]`.
    

#### Klase karaktera

U ranijem primeru regularnog izraza telefonskog broja saznali ste da `\d` može predstavljati bilo koju numeričku cifru. Odnosno, `\d` je skraćenica za regularni 
izraz `(0|1|2|3|4|5|6|7|8|9)`. Postoji mnogo takvih stenografskih klasa znakova, kako je prikazano u tabeli 7-1.

Skraćeni kodovi za uobičajene klase karaktera

Skraćenica za klasu karaktera |  Reprezentuje
:---------------------------:| :-----------------
**\d**                        | Bilo koja numerička cifra od 0 do 9
**\D**                        | Bilo koji karakter koji *nije* numerička cifra od 0 do 9
**\w**                        | Bilo koje slovo, numerička cifra ili karakter za indeks _  (Shvatite kao karaktere kojim formiramo reči)
**\W**                        | Bilo koji karakter koji *nije* slovo, numerička cifra ili karakter za indeks _
**\s**                        | Bilo koji razmak (belina), tabulator ili karakter za kraj reda (Shvatite kao znakovi za razmak)
**\S**                        | Bilo koji karakter koji *nije* razmak (belina), tabulator ili karakter za kraj reda

Klase znakova su lepe za skraćivanje regularnih izraza. Klasa znakova `[0-5]` podudaraće se samo sa brojevima od 0 do 5; ovo je mnogo kraće od kucanja 
`(0|1|2|3|4|5)`. Imajte na umu da `\d` odgovara ciframa, a `\w` podudara se sa ciframa, slovima i donjim crtama, to ne postoji stenografska klasa znakova koja se 
podudara samo sa slovima. (Iako možete da koristite klasu znakova `[a-zA-Z]`, kao što je objašnjeno u nastavku.)

Na primer, u interaktivnu ćeliju unesite sledeće:

In [90]:
pgRegex = re.compile(r'\d+\s\w+')  #cifra 1+ puta, praznina, tipografski znak 1+ puta

In [93]:
pgRegex.findall('12 gusaka, 11 pataka, 10 ovaca, 9 krava, 8 svinja, 7 koza, 6 ćurki, 5 konja, 4 magarca, 3 pileta, 2 goluba, 1 zec')

['12 gusaka',
 '11 pataka',
 '10 ovaca',
 '9 krava',
 '8 svinja',
 '7 koza',
 '6 ćurki',
 '5 konja',
 '4 magarca',
 '3 pileta',
 '2 goluba',
 '1 zec']

Regularni izraz `\d+\s\w+` poklapa se sa tekstom koji ima jednu ili više numeričkih cifara (`\d+`), a iza njega sledi razmak (`\s`), praćeni jednim ili više slova/cifara/donjih crta (`\w+`). Metoda `findall()` vraća sve podudarajuće stringove regularnog izraza na listi.

#### Konstruisanje sopstvenih klasa karaktera

Postoje slučajevi kada želite da podudarite/uparite skup znakova, ali stenografske klase znakova (`\d`, `\w`, `\s`, i tako dalje) su preširoke. Možete da definišete sopstvenu klasu znakova pomoću uglastih zagrada. Na primer, klasa znakova `[aeiouAEIOU]` podudaraće se sa bilo kojim samoglasnikom, malim i velikim. 

Unesite sledeće u interaktivnu ćeliju:

In [96]:
samoglasniciRegex = re.compile(r'[aeiouAEIOU]')

In [102]:
samoglasniciRegex.findall('Na Drini ćuprija. Ivo Andrić.')

['a', 'i', 'i', 'u', 'i', 'a', 'I', 'o', 'A', 'i']

Opsege slova ili brojeva možete da dodate i pomoću crtice. Na primer, klasa znakova `[a-zA-Z0-9]` će odgovarati svim malim slovima, velikim slovima i brojevima.

Imajte na umu da se u uglastim zagradama normalni simboli regularnih izraza ne tumače kao takvi. To znači da ne morate da izbegnete znakove `.`, `*`,`?` ili `(` `)` sa prethodnom kosom kosom crtom. Na primer, klasa znakova `[0-5.]` podudaraće se sa ciframa od 0 do 5 i tačkom. Ne morate to pisati kao `[0-5 \.]`.

Postavljanjem znaka `^`(kvačica) odmah nakon uvodne zagrade klase znakova, možete da napravite *negativnu klasu karaktera*. Negativna klasa znakova podudaraće se sa svim znakovima koji nisu u klasi znakova. Na primer, u interaktivnu ćeliju unesite sledeće:

In [103]:
suglasniciRegex = re.compile(r'[^aeiouAEIOU]')

In [104]:
suglasniciRegex.findall('Na Drini ćuprija. Ivo Andrić.')

['N',
 ' ',
 'D',
 'r',
 'n',
 ' ',
 'ć',
 'p',
 'r',
 'j',
 '.',
 ' ',
 'v',
 ' ',
 'n',
 'd',
 'r',
 'ć',
 '.']

Sada, umesto uparivanja svakog samoglasnika, uparuje se svaki karakter koji nije samoglasnik.

#### Karakteri `$` (dolar) i `^` (kvačica)

Možete koristiti kvačicu (`^`) na početku regularnog izraza da naznačite da se podudaranje mora dogoditi na početku traženog teksta. Isto tako, na kraj regularnog izraza možete staviti znak dolar (`$`) da biste označili da string mora da se završava datim šablonom regularnog izraza. A možete zajedno koristiti `^` i `$` da biste označili da ceo string mora odgovarati regularnom izrazu - to jest, nije dovoljno da se podudaranje desi na nekom delu stringa.

Na primer, regularni izraz `r'^ Dobar dan'` podudara se sa stringovima koji počinju sa `'Dobar dan'`. Unesite sledeće u interaktivnu ćeliju:

In [111]:
pocetak = re.compile(r'^Marija')

In [112]:
pocetak.search('Marija ide na posao')

<re.Match object; span=(0, 6), match='Marija'>

In [113]:
pocetak.search('Dobar dan Marija.') == None

True

Regularni izraz `r'\d$'` podudara se sa stringovima koji se završavaju numeričkim znakom od 0 do 9. Unesite sledeće u interaktivnu ćeliju:

In [114]:
ZavrsetakBrojem = re.compile(r'\d$')

In [115]:
ZavrsetakBrojem.search('Na lotou je izvučen broj 14')

<re.Match object; span=(26, 27), match='4'>

In [116]:
ZavrsetakBrojem.search('Na lotou je izvučen broj dva.') == None

True

Regularni izraz `r'^\d+$'` podudara se sa stringovima koji počinju i završavaju se jednim ili više numeričkih znakova. U interaktivnu ćeliju unesite sledeće:

In [117]:
sveBrojevi = re.compile(r'^\d+$')

In [118]:
sveBrojevi.search('1234567890')

<re.Match object; span=(0, 10), match='1234567890'>

In [119]:
sveBrojevi.search('12345xyz67890') == None

True

In [120]:
sveBrojevi.search('12 34567890') == None

True

Poslednja dva poziva `search()` u prethodnom primeru pokazuju kako se ceo string mora podudarati sa regularnim izrazom ako se koriste `^` i `$`.

#### Džoker `.` karakter

Znak za tačku (`.`) u regularnom izrazu naziva se zamenljivim znakom i podudaraće se sa bilo kojim znakom, osim sa novom linijom. Na primer, u interaktivnu ćeliju unesite sledeće:

In [135]:
laRegex = re.compile(r'.la')

In [136]:
laRegex.findall('Mila je srela Pavla i Nikolu kod sela gde je počela tiha rasprava oko stakla.')

['ila', 'ela', 'vla', 'ela', 'ela', 'kla']

Imajte na umu da će se znak za tačku podudarati sa samo jednim znakom. Da biste napravili podudaranje sa stvarnom tačkom koristite `\.` (escape character `\`)

#### Uparivanje svega sa tačka-zvezda

Ponekad ćete želeti da uparujete/podudarate sa svim i svačim. Na primer, recimo da želite da se podudaranje sa stringom „Prezime:“, praćeno bilo kojim tekstom, zatim „Prezime:“, a zatim ponovo bilo šta. Tada može da se koristi tačka-zvezdu (`.*`) da se kaže za to „bilo šta“. Zapamtite da tačkasti znak znači „bilo koji pojedinačni znak osim novog reda “, a zvezdasti znak znači„ nula ili više prethodnog znaka “. U interaktivnu ćeliju unesite sledeće:

In [139]:
imeRegex = re.compile(r'Ime: (.*) Prezime: (.*)')

In [140]:
dr = imeRegex.search('Ime: Dragan Prezime: Azdejković')

In [141]:
dr.group(1)

'Dragan'

In [142]:
dr.group(2)

'Azdejković'

Tačka-zvezda koristi pohlepni režim: Uvek će pokušati da uskladi što više teksta. Da se podudara sa bilo kojim tekstom na nepohlepni način, koristite tačku, zvezdicu i znak pitanja (. *?). Kao kod zagrada, znak pitanja govori Pajtonu da se podudara na ne-pohlepan način.

U interaktivnu ćeliju unesite sledeće da biste videli razliku između pohlepnih i ne pohlepnih verzija:

In [143]:
nePohlepniRegex = re.compile(r'<.*?>')

In [144]:
dr = nePohlepniRegex.search('<Piletina> za ručak.>')

In [145]:
dr.group()

'<Piletina>'

In [146]:
pohlepniRegex = re.compile(r'<.*>')

In [147]:
dr = pohlepniRegex.search('<Piletina> za ručak.>')

In [148]:
dr.group()

'<Piletina> za ručak.>'

Oba regeksa se približno prevode u „Podudaraju se zagrada ugla otvaranja, praćena bilo čime, a zatim zatvarajući ugaona zagrada“, ali string `'<Piletina> za ručak.>'` ima dva moguća podudaranja za zatvaranje ugaonih zagrada. U ne pohlepnoj verziji regularnog izraza, Pajton podudara sa najkraćim mogućim stringom: `'<Piletina>'`. U pohlepnoj verziji Pajton se podudara sa najdužim mogućim nizom: `'<Piletina> za ručak.>'`.

#### Uparivanje nove linije sa tačkastim karakterom

Tačka-zvezda će se podudarati sa svim osim sa novom linijom. Predajom `re.DOTALL` kao drugog argumenta `re.compile()`, možete učiniti da se tačkasti znak podudara 
sa svim znakovima, uključujući i znak novog reda.

U interaktivnu ćeliju unesite sledeće:

In [149]:
NovaLinijaRegex = re.compile('.*')

In [150]:
NovaLinijaRegex.search('Služiti javnom poverenju.\nŠtititi nevine.\nPoštovati zakon.').group()

'Služiti javnom poverenju.'

In [152]:
NovaLinijaRegex = re.compile('.*', re.DOTALL)

In [153]:
newlineRegex.search('Služiti javnom poverenju.\nŠtititi nevine.\nPoštovati zakon.').group()

'Služiti javnom poverenju.\nŠtititi nevine.\nPoštovati zakon.'

Regeks `NovaLinijaRegex`, koji nema prosleđeno `re.DOTALL` metodu `re.compile`, uparivaće sve da prvog znaka za novu liniju, dok `NovaLinijaRegex`, koji ima prosleđeno `re.DOTALL` metodu `re.compile`, uparivaće sve. To je razlog zašto pozivanjem `NovaLinijaRegex.search()` uparuje ceo string uključujući i karakter za kraj reda.

#### Pregled Regeks simbola

Ovo poglavlje pokrivalo je puno notacija, pa evo kratkog pregleda onoga šta ste naučili o osnovnoj sintaksi regularnog izraza: 
    
- `?` zahteva podudaranje nula ili jednom sa grupom koja prethodi. 
- Znak `*` zahteva podudaranje nula ili više puta sa grupom koja prethodi. 
- Znak `+` zahteva podudaranje jednom ili više puta sa grupom koja prethodi. 
- Oznaka `{n}` zahteva podudaranje tačno $n$ puta sa grupom koja prethodi. 
- Oznaka `{n ,}` zahteva podudaranje najmanje $n$ puta sa grupom koja prethodi. 
- `{, m}` zahteva podudaranje najviše $m$ puta sa grupom koja prethodi. 
- `{n, m}` zahteva podudaranje najmanje $n$ i najviše $m$ puta sa grupom koja prethodi. 
- `{n, m}?` ili *? ili +? izvršava ne pohlepno podudaranje prethodne grupe. 
- `^spam` znači da string mora počinjati sa `spam`. 
- `spam\$` znači da se string mora završiti sa `spam`. 
- `.` podudara se sa bilo kojim znakom, osim znakova za novi red. 
- `\d`, `\w` i `\s` odgovaraju cifri, slovu ili razmaku, odnosno. 
- `\D`, `\W` i `\S` odgovaraju bilo čemu osim cifri, slovu ili razmaku znak. 
- `[abc]` se podudara sa bilo kojim znakom između zagrada (poput *a*, *b* ili *c*). 
- `[^abc]` se podudara sa bilo kojim znakom koji nije između zagrada.

#### Uparivanje osetljivo na veličinu slova

Obično se regularni izrazi podudaraju s tekstom sa tačnim slovima koje navedete. Na primer, sledeći regularni izrazi se podudaraju sa potpuno različitim stringovima:

In [121]:
regex1 = re.compile('RoboCop')
regex2 = re.compile('ROBOCOP')
regex3 = re.compile('robOcop')
regex4 = re.compile('RobocOp')

Ali ponekad vam je stalo samo da se slova podudaraju, ne brinući da li su velika ili mala. Da biste učinili da vaš regeks ne razlikuje velika i mala slova, možete 
dodati `re.IGNORECASE` ili `re.I` kao drugi argument u `re.compile()`. U interaktivnu ćeliju unesite sledeće:

In [122]:
robocop = re.compile(r'robocop', re.I)

In [123]:
robocop.search('RoboCop is part man, part machine, all cop.').group()

'RoboCop'

In [124]:
robocop.search('ROBOCOP protects the innocent.').group()

'ROBOCOP'

In [125]:
robocop.search('Al, why does your programming book talk about robocop so much?').group()

'robocop'

#### Zamena stringova metodom `sub()`

Regularni izrazi mogu ne samo da pronađu tekstualne obrasce, već mogu i zameniti novi tekst umesto tih obrazaca. Metodi `sub()` za regeks objekte prosleđuju se dva argumenta. Prvi argument je string koji zamenjuje bilo koja podudaranja. Drugi je string za regularni izraz. Metoda `sub()` vraća string sa primenjenim zamenama.

Na primer, u interaktivnu ćeliju unesite sledeće:

In [174]:
imenaRegex = re.compile(r'\bagent\w*\s+\w+', re.IGNORECASE)

In [175]:
imenaRegex.sub('TAJNA', 'Agent Aleksandar i agent Marijа su otišli u Beograd.')

'TAJNA i TAJNA su otišli u Beograd.'

Ponekad ćete možda morati da koristite sam tekst koji se podudara kao deo zamene. U prvom argumentu za `sub()` možete otkucati `\1`, `\2`, `\3` i tako dalje, što 
znači „Unesite tekst grupe 1, 2, 3 i tako dalje, u zamenu.“

Na primer, recimo da želite da cenzurišete imena tajnih agenata tako što ćete prikazati samo prva slova njihovih imena. Da biste to uradili, možete koristiti 
regularni izraz `Agent`, `(\w)\w*`, i predati `r'\1****'` kao prvi argument `sub()`. `\1` u tom stringu biće zamenjen bilo kojim tekstom koji se podudara sa 
grupom 1 - to jest, `(\w)` grupom regularnog izraza.

In [172]:
agentiRegex = re.compile(r'agent\w*\s+(\w)\w*', re.IGNORECASE)

In [177]:
agentiRegex.sub(r'\1****', "Agent Marija je  javila agentu Aleksandru da je dvostruki agent Tara.")

'M**** je  javila A**** da je dvostruki T****.'

#### Upravljanje složenim regeksima

Regularni izrazi su u redu ako je obrazac teksta koji treba da se podudara jednostavan. Ali podudaranje složenih tekstualnih obrazaca može zahtevati dugačke, 
zamršene regularne izraze. To možete ublažiti rekavši funkciji `re.compile()` da zanemari razmake i komentare unutar stringa regularnog izraza. Ovaj „opširan režim“ 
može se omogućiti dodavanjem promenljive `re.VERBOSE` kao drugog argumenta u `re.compile()`.

Sada umesto ovako čitljivog regularnog izraza:

In [178]:
telRegex = re.compile(r'((\d{3}|\(\d{3}\))?(\s|-|\.)?\d{3}(\s|-|\.)\d{4}(\s*(ext|x|ext.)\s*\d{2,5})?)')

regularni izraz možete proširiti na više redova sa komentarima poput ovog:

In [179]:
telRegex = re.compile(r'''(
     (\d{3}|\(\d{3}\))?            # kod zone
     (\s|-|\.)?                    # separator
     \d{3}                         # prve tri cifre
     (\s|-|\.)                     # separator
     \d{4}                         # poslednje tri cifre
    (\s*(ext|x|ext.)\s*\d{2,5})?   # produženje
     )''', re.VERBOSE)

Obratite pažnju na to kako prethodni primer koristi sintaksu sa trostrukim navodnicima (`'''`) za stvaranje višerednog stringa, tako da možete proširiti definiciju regularnog izraza na više linija, čineći ga mnogo čitljivijim.

Pravila za komentare unutar stringa regularnog izraza ista su kao i kod regularnog Pajtona: simbol `#` i sve nakon njega do kraja reda se zanemaruju. Takođe, dodatni razmaci unutar višerednog stringa za regularni izraz ne smatraju se delom šablona teksta koji treba podudarati. Ovo vam omogućava da organizujete regularni izraz tako da ga je lakše čitati.

#### Kombinovanje `re.IGONORECASE`, `re.DOTALL` i `re.VERBOSE`

Šta ako želite da koristite `re.VERBOSE` za pisanje komentara u svoj regularni izraz, ali takođe želite da koristite `re.IGNORECASE` da biste zanemarili upotrebu 
velikih slova? Nažalost, funkcija `re.compile()` uzima samo jednu vrednost kao drugi argument. Ovo ograničenje možete zaobići kombinovanjem promenljivih 
`re.IGNORECASE`, `re.DOTALL` i `re.VERBOSE` koristeći znak cevi (|), koji je u ovom kontekstu poznat kao bitni ili operator.

Dakle, ako želite regularni izraz koji ne razlikuje velika i mala slova i uključuje nove redove koji se podudaraju sa tačkastim znakom, formirali biste svoj poziv 
`re.compile()` ovako:

In [180]:
nekaRegexVrednost = re.compile('foo', re.IGNORECASE | re.DOTALL)

Uključivanje sve tri opcije kao drugi argument bi izgleadalo ovako:

In [181]:
nekaRegexVrednost = re.compile('foo', re.IGNORECASE | re.DOTALL | re.VERBOSE)

Ova sintaksa je pomalo staromodna i potiče iz ranih verzija Pajtona. Detalji bitnih operatora su izvan opsega ove knjige, ali potražite resurse na 
https://nostarch.com/automatestuff2/ za više informacija. Takođe možete proslediti druge opcije za drugi argument; oni su neuobičajeni, 
ali više o njima možete pročitati i u resursima.

<br>

### Vežbanja

::::{tab-set}

:::{tab-item} ✍️ Zadatak 1

Pronađi sve brojeve u datom stringu:

```python
tekst = "Računi su 120, 45 i 789 dinara."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Računi su 120, 45 i 789 dinara."

regex = re.compile(r'\d+')

rezultat = regex.findall(tekst)

print(rezultat)
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 2

Pronađi sve reči koje počinju velikim slovom:

```python
tekst = "Marko i Ana su otišli u Beograd."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Marko i Ana su otišli u Beograd."

regex = re.compile(r'\b[A-Z][a-z]+')

rezultat = regex.findall(tekst)

print(rezultat)
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 3

Zameni sve brojeve u tekstu sa slovom „X“:

```python
tekst = "Cena je 500 dinara, popust 20%."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Cena je 500 dinara, popust 20%."

regex = re.compile(r'\d+')

rezultat = regex.sub('X', tekst)

print(rezultat)
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 4

Izdvoji sve reči koje se završavaju na „la“:

```python
tekst = "Mila je videla stakla i ogledala."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Mila je videla stakla i ogledala."

regex = re.compile(r'\w*la\b')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 5

Pronađi sve trocifrene brojeve:

```python
tekst = "Brojevi su 12, 123, 4567 i 890."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Brojevi su 12, 123, 4567 i 890."

regex = re.compile(r'\b\d{3}\b')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 7

Izdvoji sve hashtagove:

```python
tekst = "Program #python je bolji od #java i #csharp"
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Program #python je bolji od #java i #csharp"

regex = re.compile(r'#\w+')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 8

Izdvoji sve reči koje imaju tačno 5 slova:

```python
tekst = "Marko ide kući danas i nosi knjigu."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Marko ide kući danas i nosi knjigu."

regex = re.compile(r'\b\w{5}\b')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 9

Zameni sve samoglasnike sa „*“:

```python
tekst = "Programiranje je zanimljivo."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Programiranje je zanimljivo."

regex = re.compile(r'[aeiouAEIOU]')

print(regex.sub('*', tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 10

Izdvoji sve datume formata dd-mm-yyyy:

```python
tekst = "Datumi su 12-05-2023 i 01-01-2024."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Datumi su 12-05-2023 i 01-01-2024."

regex = re.compile(r'\d{2}-\d{2}-\d{4}')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 11

Maskiraj sve brojeve osim poslednje dve cifre:

```python
tekst = "Kartica: 12345678"
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Kartica: 12345678"

regex = re.compile(r'\d(?=\d{2})')

print(regex.sub('*', tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 12

Izdvoji sva imena i zameni ih inicijalima (npr. Marko → M.):

```python
tekst = "Marko i Ana su otišli u Beograd."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Marko i Ana su otišli u Beograd."

regex = re.compile(r'\b([A-Z])[a-z]+')

print(regex.sub(r'\1.', tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 13

Izdvoji sve brojeve koji se nalaze iza simbola „€“, ali ne i ostale brojeve:

```python
tekst = "Cena je €100, popust je 20%, a nova cena €80."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Cena je €100, popust je 20%, a nova cena €80."

regex = re.compile(r'(?<=€)\d+')

print(regex.findall(tekst))
```

:::

::::

:::{tab-item} ✍️ Zadatak 6

Izdvoji sve email adrese:

```python
tekst = "Kontakt: test@gmail.com i info@yahoo.com"
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Kontakt: test@gmail.com i info@yahoo.com"

regex = re.compile(r'\w+@\w+\.\w+')

print(regex.findall(tekst))
```

:::

::::

::::{tab-set}

:::{tab-item} ✍️ Zadatak 14

Zameni sve duple reči (npr. „je je“, „da da“) jednom pojavom reči:

```python
tekst = "Ovo je je test da da proverimo regex regex."
```

:::

:::{tab-item} Rešenje

```python
import re

tekst = "Ovo je je test da da proverimo regex regex."

regex = re.compile(r'\b(\w+)\s+\1\b', re.IGNORECASE)

print(regex.sub(r'\1', tekst))
```

:::

::::

### Zadaci i problemi

**⚡️Zadatak 1.**

Ako je `numRegex = re.compile (r'\d+')`, šta će vratiti?
```python
numRegex.sub ('Ks', '12 bubnjara, 11 lula, pet prstenova, 3 kokoši ')
```

**⚡️Zadatak 2.**

Kako biste napisali regularni izraz koji odgovara broju i zapetama za svake tri cifre? Mora odgovarati sledećem: ‘42’, ‘1,234’, ‘6,368,745’, ali ne i sledećem: '12, 34,567 '(koja ima samo dve cifre između zareza), ’ 1234 '(kojoj nedostaju zarezi)

**⚡️Zadatak 3.**

Kako biste napisali regularni izraz koji se podudara sa punim imenom nekoga ko se preziva Vatanabe? Možete pretpostaviti da će prvo ime koje dolazi pre njega uvek biti jedna reč koja počinje velikim slovom. Redovni izraz mora odgovarati sledećem: 'Haruto Vatanabe', 'Alice Vatanabe', 'RoboCop Vatanabe', ali ne i sledećem: 'haruto Vatanabe' (gde ime nije napisano velikim slovom), 'Mr. Vatanabe '(gde prethodna reč ima neslovni karakter), ' Vatanabe '(bez imena), ' Haruto vatanabe '(gde Vatanabe nije napisana velikim slovom)

**⚡️Zadatak 3.**

Kako biste napisali regularni izraz koji se podudara sa rečenicom u kojoj je prva reč Alice, Bob ili Carol; druga reč je ili jede, ljubimce ili baca; treća reč su jabuke, mačke ili lopte; a rečenica se završava tačkom? Ovaj regularni izraz ne bi trebalo da razlikuje velika i mala slova. Mora se podudarati sa sledećim: 'Alice jede jabuke.', 'Bob maci mačke.',  'Carol baca lopte', 'Alice baca jabuke.', 'BOB JE MAČKA', ali ne i sledeće: 'RoboCop jede jabuke.', 'ALICE BACA NOGU.', 
'Carol jede 7 mačaka'.

**⚡️Zadatak 4.**

Napišite regularni izraz koji može otkriti datume u formatu `DD/MM/GGGG`. Pretpostavimo da se dani kreću od 01 do 31, meseci od 01 do 12, a godine od 1000 do 2999. 
Imajte na umu da ako je dan ili mesec jednocifren, imaće vodeću nulu.

Regularni izraz ne mora da otkriva tačne dane za svaki mesec ili za prestupnu godinu; prihvatiće nepostojeće datume poput 31/02/2020 ili 31/04/2021. Zatim te 
stringove sačuvajte u promenljive sa imenom mesec, dan i godina i napišite dodatni kod koji može otkriti da li je to valjani datum. April, jun, septembar i novembar 
imaju 30 dana, februar 28 dana, a ostatak meseci 31 dan. Februar ima 29 dana u prestupnoj godini. Prestupne godine su svake godine ravnomerno deljive sa 4, 
osim godina ravnomerno deljivih sa 100, osim ako je godina takođe ravnomerno deljiva sa 400. Obratite pažnju kako ovaj proračun onemogućava pravljenje regularnog 
izraza razumne veličine koji može otkriti valjani datum.

**⚡️Zadatak 5.**

Napišite funkciju koja koristi regularne izraze kako bi bila sigurna da je string lozinke koji je prosleđen jak. Jaka lozinka je definisana kao lozinka koja ima 
najmanje osam znakova, sadrži velika i mala slova i ima najmanje jednu cifru. Možda ćete trebati da testirate string na osnovu višestrukih regularnih izraza da 
biste potvrdili njegovu snagu.

**⚡️Zadatak 6.**

Napišite funkciju koja uzima string i radi isto što i string metoda `strip()`. Ako se ne dodaju drugi argumenti osim stringa za uklanjanje, razmaci će se ukloniti 
s početka i kraja stringa. U suprotnom, znakovi navedeni u drugom argumentu funkcije biće uklonjeni iz stringa.

<br>

### Studije slučaja
<br>

🧩 **Ekstraktor e-mail adrese**

Napraviti program koji omogućava korisniku da nalepi tekst kopiran sa web stranice, a zatim automatski pronalazi sve email adrese koje se u tom tekstu pojavljuju.

Program treba da:
1. učita tekst koji korisnik unosi,
2. pronađe sve email adrese,
3. prikaže listu pronađenih adresa.

🧩 **Hashtagovi**

Napraviti program koji omogućava korisniku da nalepi tekst objave sa društvene mreže, a zatim pronalazi sve hashtagove.

Program treba da:
1. učita tekst koji korisnik nalepi,
2. izdvoji sve izraze koji počinju znakom `#`,
3. prikaže pronađene hashtagove.